# Malaria & Infectious Disease Burden: Africa vs USA

**Domain:** Health Sector | **Focus:** Sub-Saharan Africa & the United States

This notebook explores the disparity in malaria burden and broader health indicators
between ten Sub-Saharan African focus countries (Kenya, Ethiopia, Nigeria, Ghana,
South Africa, Uganda, Tanzania, Mozambique, Zambia, Malawi) and the United States.

Data comes from the **WHO Global Health Observatory** (malaria incidence & deaths) and
the **World Bank** (health expenditure, life expectancy, under-5 mortality), processed
by the project's ETL pipeline into `data/processed/` and `data/health_disease_burden.db`.


In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Resolve the project root whether the notebook is run from notebooks/ or root.
CWD = os.getcwd()
ROOT = CWD if os.path.isdir(os.path.join(CWD, "data")) else os.path.dirname(CWD)
RAW = os.path.join(ROOT, "data", "raw")
PROCESSED = os.path.join(ROOT, "data", "processed")
DB_PATH = os.path.join(ROOT, "data", "health_disease_burden.db")
print("Project root:", ROOT)


## 1. Load the data

We load the processed master datasets when available and fall back to the raw CSVs.
The SQLite database is used when it has already been built by `src/loading/load.py`.


In [ ]:
def load_master():
    master_path = os.path.join(PROCESSED, "malaria_master.csv")
    health_path = os.path.join(PROCESSED, "health_indicators.csv")
    if os.path.exists(master_path) and os.path.exists(health_path):
        return pd.read_csv(master_path), pd.read_csv(health_path)
    # Fallback: build lightweight views straight from raw CSVs.
    inc = pd.read_csv(os.path.join(RAW, "malaria_incidence_africa.csv"))
    focus = ["KEN","ETH","NGA","GHA","ZAF","UGA","TZA","MOZ","ZMB","MWI"]
    inc = inc[inc.country_code.isin(focus)].copy()
    inc["region"] = "Africa"
    he = pd.read_csv(os.path.join(RAW, "health_expenditure_pct_gdp.csv"))
    le = pd.read_csv(os.path.join(RAW, "life_expectancy.csv"))
    u5 = pd.read_csv(os.path.join(RAW, "under5_mortality.csv"))
    health = he.merge(le, on=["country_code","country_name","year"], how="outer") \
               .merge(u5, on=["country_code","country_name","year"], how="outer")
    health["region"] = np.where(health.country_code == "USA", "USA", "Africa")
    return inc, health

malaria, health = load_master()
print("malaria_master:", malaria.shape)
print("health_indicators:", health.shape)
malaria.head()


## 2. Exploratory data analysis

A quick look at coverage, year ranges and summary statistics across the focus countries.


In [ ]:
print("Malaria years:", int(malaria.year.min()), "-", int(malaria.year.max()))
print("Countries:", sorted(malaria.country_code.unique()))
print()
print("Incidence per 1,000 — summary:")
print(malaria["incidence_per_1000"].describe().round(2))
print()
print("Health indicators regions:", health.region.value_counts().to_dict())
health.describe(numeric_only=True).round(2)


## 3. Visualization 1 — Top 10 African countries by malaria incidence (latest year)

Which focus countries carry the heaviest malaria burden in the most recent year available?


In [ ]:
latest_year = int(malaria.year.max())
latest = malaria[malaria.year == latest_year].copy()
# Some countries may not report the very latest year; take each country's latest.
latest = malaria.sort_values("year").groupby("country_code").tail(1)
name_col = "country_name" if "country_name" in latest.columns else "country_code"
top10 = latest.sort_values("incidence_per_1000", ascending=False).head(10)

plt.figure()
sns.barplot(data=top10, x="incidence_per_1000", y=name_col, palette="Reds_r")
plt.title(f"Top African countries by malaria incidence (latest reported year)")
plt.xlabel("Incidence per 1,000 population at risk")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 4. Visualization 2 — Incidence trend 2010–2024 for the top 5 countries

Trends reveal whether high-burden countries are improving or stagnating over time.


In [ ]:
top5_codes = top10.head(5)["country_code"].tolist()
trend = malaria[malaria.country_code.isin(top5_codes)].sort_values("year")

plt.figure()
for code, grp in trend.groupby("country_code"):
    label = grp[name_col].iloc[0] if name_col in grp else code
    plt.plot(grp.year, grp.incidence_per_1000, marker="o", label=label)
plt.title("Malaria incidence trend — top 5 high-burden countries")
plt.xlabel("Year")
plt.ylabel("Incidence per 1,000")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Visualization 3 — Health expenditure vs life expectancy (Africa + USA)

An interactive scatter (plotly) comparing how health investment relates to life expectancy.
The USA appears as a clear high-expenditure, high-life-expectancy outlier.


In [ ]:
scatter_df = health.dropna(subset=["health_expenditure_pct_gdp", "life_expectancy_years"]).copy()
fig = px.scatter(
    scatter_df,
    x="health_expenditure_pct_gdp",
    y="life_expectancy_years",
    color="region",
    hover_name="country_name" if "country_name" in scatter_df else "country_code",
    hover_data=["year"],
    title="Health expenditure (% GDP) vs life expectancy",
    labels={
        "health_expenditure_pct_gdp": "Health expenditure (% of GDP)",
        "life_expectancy_years": "Life expectancy (years)",
    },
)
fig.show()


## 6. Visualization 4 — Heatmap of malaria incidence by country and year

A seaborn heatmap shows the intensity of malaria incidence across countries and years
at a glance, highlighting persistent hotspots.


In [ ]:
pivot = malaria.pivot_table(
    index="country_code", columns="year", values="incidence_per_1000", aggfunc="mean"
)
plt.figure(figsize=(12, 6))
sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.3, linecolor="white")
plt.title("Malaria incidence per 1,000 — country x year")
plt.xlabel("Year")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


## 7. Visualization 5 — Under-5 mortality: Africa vs USA

Under-5 mortality is a sensitive marker of overall child-health conditions and infectious
disease burden. We compare the African focus-country average against the USA.


In [ ]:
u5 = health.dropna(subset=["under5_mortality_per_1000"]).copy()
u5_latest = u5.sort_values("year").groupby("country_code").tail(1)
u5_summary = (
    u5_latest.groupby("region")["under5_mortality_per_1000"].mean().reset_index()
)

plt.figure()
sns.barplot(data=u5_summary, x="region", y="under5_mortality_per_1000", palette="Blues_r")
plt.title("Under-5 mortality (latest year) — Africa avg vs USA")
plt.xlabel("")
plt.ylabel("Under-5 mortality per 1,000 live births")
plt.tight_layout()
plt.show()


## 8. Visualization 6 — Life expectancy trends: Africa vs USA

Finally, we track the life-expectancy gap over time between the African focus-country
average and the United States.


In [ ]:
le = health.dropna(subset=["life_expectancy_years"]).copy()
le_trend = le.groupby(["region", "year"])["life_expectancy_years"].mean().reset_index()

plt.figure()
for region, grp in le_trend.groupby("region"):
    plt.plot(grp.year, grp.life_expectancy_years, marker="o", label=region)
plt.title("Life expectancy trend — Africa average vs USA")
plt.xlabel("Year")
plt.ylabel("Life expectancy (years)")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Conclusions

- **Africa carries a vastly higher malaria burden.** The focus countries report
  substantial malaria incidence and deaths every year, while the USA has no endemic
  transmission — consistent with the WHO estimate that Africa accounts for 95%+ of the
  global malaria burden.
- **Burden is concentrated.** Countries such as Nigeria, Mozambique and Uganda show the
  highest incidence, whereas South Africa sits far lower thanks to sustained control.
- **Health investment tracks better outcomes.** The scatter of health expenditure vs
  life expectancy — and the expenditure/under-5-mortality relationship — supports an
  inverse link between health spending and disease burden.
- **A wide, slow-closing gap.** Life expectancy in the focus African countries trails
  the USA by many years, and under-5 mortality remains several times higher — underscoring
  the scale of the infectious-disease and health-system disparity.
